In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# possible locations for the external prediction files
base_dirs = [
    "../input/melanoma-dif-sub",
    "/kaggle/input/melanoma-dif-sub",
    "input/melanoma-dif-sub",
    "melanoma-dif-sub",
]

sub_filenames = [
    "pl_0.936.csv",
    "pl_0.940.csv",
    "sub_EfficientNetB2_384.csv",
    "sub_EfficientNetB3_384.csv",
    "sub_EfficientNetB3_384_v2.csv",
]


def resolve_path(fname):
    """Return the first existing full path for fname in known base directories."""
    for d in base_dirs:
        p = os.path.join(d, fname)
        if os.path.isfile(p):
            return p
    # fallback – maybe an absolute path was given directly
    return fname if os.path.isfile(fname) else None


def load_or_placeholder(path, placeholder_names):
    """Load a CSV if it exists; otherwise create a neutral placeholder."""
    if path is None:
        # file not found → placeholder
        return pd.DataFrame(
            {
                "image_name": placeholder_names,
                "target": np.full(len(placeholder_names), 0.5),
            }
        )
    try:
        sub = pd.read_csv(path)
    except Exception:
        sub = pd.DataFrame(
            {
                "image_name": placeholder_names,
                "target": np.full(len(placeholder_names), 0.5),
            }
        )
    return sub


def get_test_image_names():
    possible_paths = [
        "../input/siim-isic-melanoma-classification/test.csv",
        "/kaggle/input/siim-isic-melanoma-classification/test.csv",
        "input/siim-isic-melanoma-classification/test.csv",
        "test.csv",
        "sample_submission.csv",
        "data/sample_submission.csv",
    ]
    for p in possible_paths:
        if os.path.exists(p):
            try:
                df = pd.read_csv(p)
                if "image_name" in df.columns:
                    return df["image_name"].tolist()
            except Exception:
                continue
    return []


test_image_names = get_test_image_names()

# resolve each external CSV path and load (or create placeholder)
resolved_paths = [resolve_path(f) for f in sub_filenames]
sub1 = load_or_placeholder(resolved_paths[0], test_image_names)
sub2 = load_or_placeholder(resolved_paths[1], test_image_names)
sub3 = load_or_placeholder(resolved_paths[2], test_image_names)
sub4 = load_or_placeholder(resolved_paths[3], test_image_names)
sub5 = load_or_placeholder(resolved_paths[4], test_image_names)


def min_max_scale(df):
    col = df["target"]
    rng = col.max() - col.min()
    if rng < 1e-9:  # constant column – keep neutral prediction
        df["target"] = 0.5
    else:
        df["target"] = (col - col.min()) / rng
    return df


sub1 = min_max_scale(sub1)
sub2 = min_max_scale(sub2)
sub3 = min_max_scale(sub3)
sub4 = min_max_scale(sub4)
sub5 = min_max_scale(sub5)

sub1.columns = ["image_name", "target1"]
sub2.columns = ["image_name", "target2"]
sub3.columns = ["image_name", "target3"]
sub4.columns = ["image_name", "target4"]
sub5.columns = ["image_name", "target5"]


In [2]:
train_path_options = [
    "../input/siim-isic-melanoma-classification/train.csv",
    "/kaggle/input/siim-isic-melanoma-classification/train.csv",
    "input/siim-isic-melanoma-classification/train.csv",
    "../input/train.csv",
    "train.csv",
]
train_path = next((p for p in train_path_options if os.path.exists(p)), None)

if train_path is not None:
    train_df = pd.read_csv(train_path)

    feature_cols = [
        "sex",
        "age_approx",
        "anatom_site_general_challenge",
        "diagnosis",
        "benign_malignant",
    ]
    X = train_df[feature_cols].copy()
    y = train_df["target"]

    X["sex"] = X["sex"].fillna("unknown")
    X["anatom_site_general_challenge"] = X["anatom_site_general_challenge"].fillna(
        "unknown"
    )
    X["diagnosis"] = X["diagnosis"].fillna("unknown")
    X["benign_malignant"] = X["benign_malignant"].fillna("unknown")
    X["age_approx"] = X["age_approx"].fillna(X["age_approx"].median())

    categorical_features = [
        "sex",
        "anatom_site_general_challenge",
        "diagnosis",
        "benign_malignant",
    ]
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
        ],
        remainder="passthrough",
    )

    clf = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            (
                "model",
                LogisticRegression(
                    C=2.0,
                    max_iter=2000,
                    n_jobs=1,
                    solver="lbfgs",
                    class_weight="balanced",
                ),
            ),
        ]
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    clf.fit(X_train, y_train)

    test_path_options = [
        "../input/siim-isic-melanoma-classification/test.csv",
        "/kaggle/input/siim-isic-melanoma-classification/test.csv",
        "input/siim-isic-melanoma-classification/test.csv",
        "test.csv",
    ]
    test_path = next((p for p in test_path_options if os.path.exists(p)), None)

    if test_path is not None:
        test_df = pd.read_csv(test_path)

        X_test = pd.DataFrame()
        for col in feature_cols:
            if col in test_df.columns:
                X_test[col] = test_df[col]
            else:
                if col == "age_approx":
                    X_test[col] = X["age_approx"].median()
                else:
                    X_test[col] = "unknown"

        X_test["sex"] = X_test["sex"].fillna("unknown")
        X_test["anatom_site_general_challenge"] = X_test[
            "anatom_site_general_challenge"
        ].fillna("unknown")
        X_test["diagnosis"] = X_test["diagnosis"].fillna("unknown")
        X_test["benign_malignant"] = X_test["benign_malignant"].fillna("unknown")
        X_test["age_approx"] = X_test["age_approx"].fillna(X["age_approx"].median())

        test_pred = clf.predict_proba(X_test)[:, 1]
        sub6 = pd.DataFrame({"image_name": test_df["image_name"], "target": test_pred})
        sub6 = min_max_scale(sub6)
        sub6.columns = ["image_name", "target6"]
    else:
        sub6 = pd.DataFrame(
            {
                "image_name": test_image_names,
                "target6": np.full(len(test_image_names), 0.5),
            }
        )
else:
    sub6 = pd.DataFrame(
        {"image_name": test_image_names, "target6": np.full(len(test_image_names), 0.5)}
    )


In [3]:
f_sub = (
    sub1.merge(sub2, on="image_name")
    .merge(sub3, on="image_name")
    .merge(sub4, on="image_name")
    .merge(sub5, on="image_name")
    .merge(sub6, on="image_name")
)

# Give the internal tabular model a modest contribution while keeping the strong
# external predictions dominant.
base_weights = {
    "target1": 0.45,
    "target2": 0.45,
    "target3": 0.0,
    "target4": 0.0,
    "target5": 0.0,
    "target6": 0.10,
}

effective_weights = {}
for col, w in base_weights.items():
    if f_sub[col].std() < 1e-6:  # constant column → placeholder
        effective_weights[col] = 0.0
    else:
        effective_weights[col] = w

total_weight = sum(effective_weights.values())
if total_weight == 0:
    f_sub["target"] = 0.5
else:
    for col in effective_weights:
        effective_weights[col] /= total_weight
    f_sub["target"] = sum(f_sub[col] * w for col, w in effective_weights.items())

blend_submission = f_sub[["image_name", "target"]]
blend_submission.to_csv("blend_sub.csv", index=False)